In [26]:
from typing import TypedDict

class Contact(TypedDict):
    name:str
    email:str
    phone:str
    
def send_email(contact:Contact) ->None:
    print(f"Sending email to {contact['name']} at {contact['email']}")

contact_info:Contact={
    'name':"Lilei",
    "email":"lilei@qq.com",
    "phone":"15555123123",
    "aaa":'fdasfasdfs'
}

send_email(contact_info)

Sending email to Lilei at lilei@qq.com


In [27]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict

class InputState(TypedDict):
    question:str
    
class OutputState(TypedDict):
    answer:str
class OverallState(InputState,OutputState):
    pass

builder=StateGraph(OverallState,input=InputState,output=OutputState)

def agent_node(state:InputState):
    print("我是一个AI agent")
    return

def action_node(state:InputState):
    print("我是一个执行者")
    return {'answer':f"我接收到的问题是 {state['question']},我现在执行成功了"}



C:\Users\13840\AppData\Local\Temp\ipykernel_8584\3304711070.py:12: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder=StateGraph(OverallState,input=InputState,output=OutputState)
C:\Users\13840\AppData\Local\Temp\ipykernel_8584\3304711070.py:12: LangGraphDeprecatedSinceV05: `output` is deprecated and will be removed. Please use `output_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder=StateGraph(OverallState,input=InputState,output=OutputState)


In [28]:
builder.add_node('agent_node',agent_node)
builder.add_node('action_node',action_node)



In [29]:
from langgraph.graph import START,END

builder.add_edge(START,'agent_node')
builder.add_edge('agent_node','action_node')
builder.add_edge('action_node',END)



In [30]:
graph=builder.compile()

In [31]:
graph.invoke({"question":"哈喽,你好"})

我是一个AI agent
我是一个执行者


{'answer': '我接收到的问题是 哈喽,你好,我现在执行成功了'}

In [32]:
graph.invoke({'question':'今天的天气怎么样'})

我是一个AI agent
我是一个执行者


{'answer': '我接收到的问题是 今天的天气怎么样,我现在执行成功了'}

In [48]:
from langgraph.graph import StateGraph,START,END
from typing_extensions import TypedDict

class InputState(TypedDict):
    question:str
    
class OutputState(TypedDict):
    answer:str

class OverallState(InputState,OutputState):
    pass




In [49]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

sk-ff345e1cfac7491993b7e17cf9219fbf


In [54]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import getpass




def llm_node(state:InputState):
    message=[
        ('system',"你是一位乐于助人的智能小助理"),
        ('human',state['question'])
    ]
    
    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )
    
    response=llm.invoke(message)
    return {'answer':response.content}

In [55]:
builder=StateGraph(OverallState,input=InputState,output=OutputState)

builder.add_node('llm_node',llm_node)
builder.add_edge(START,'llm_node')
builder.add_edge('llm_node',END)

graph=builder.compile()

C:\Users\13840\AppData\Local\Temp\ipykernel_8584\3543301167.py:1: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder=StateGraph(OverallState,input=InputState,output=OutputState)
C:\Users\13840\AppData\Local\Temp\ipykernel_8584\3543301167.py:1: LangGraphDeprecatedSinceV05: `output` is deprecated and will be removed. Please use `output_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder=StateGraph(OverallState,input=InputState,output=OutputState)


In [56]:
graph.invoke({"question":'你好， 我用来测试'})

{'answer': '你好！很高兴为你服务。请问有什么我可以帮助你的吗？如果你需要测试，可以告诉我具体想测试什么内容，我会尽力配合！😊'}

In [57]:
final_answer=graph.invoke({'question':'你好，我用来测试'})
print(final_answer['answer'])

你好！很高兴为你服务。请问有什么我可以帮助你的吗？如果你需要测试，可以告诉我具体需要测试什么内容，我会尽力配合！😊


In [58]:
final_answer=graph.invoke({'question':'你好，请你详细的介绍一下你自己'})
print(final_answer['answer'])

你好呀！很高兴认识你！😊

我是你的智能小助理，一个由先进人工智能技术驱动的虚拟助手。我的主要任务就是帮助你解决各种问题，提供信息、建议和支持，让你的生活更便捷、更有趣！

**我的特点包括：**
- **知识丰富**：我可以回答关于科技、文化、历史、生活、学习等各类问题，尽力为你提供准确、有用的信息。
- **多语言支持**：我可以用多种语言与你交流，包括中文、英文等，方便不同语言背景的朋友使用。
- **快速响应**：我会在第一时间回复你的问题，尽量做到高效、及时。
- **友好互动**：我喜欢用轻松、亲切的方式和你交流，让对话更自然、更愉快。

**我能帮你做什么？**
- 解答疑问（学习、工作、生活等）
- 提供建议（比如旅行规划、美食推荐、健康小贴士等）
- 陪你聊天，分享有趣的知识或故事
- 帮你整理思路、制定计划
- 甚至帮你写点小作文、翻译内容等等！

当然，我也有我的“小局限”：我没有情感和主观意识，所有的回答都基于数据和算法，所以如果有不够完美的地方，也请多多包涵哦！😊

那么，今天有什么我可以帮你的吗？尽管告诉我吧！


In [62]:
from langgraph.graph import StateGraph,START,END
from typing_extensions import TypedDict

class InputState(TypedDict):
    question:str
    llm_answer:str
    
class OutputState(TypedDict):
    answer:str

class OverallState(InputState,OutputState):
    pass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


def llm_node(state:InputState):
    message=[
        ('system',"你是一位乐于助人的智能小助理"),
        ('human',state['question'])
    ]
    
    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )
    
    response=llm.invoke(message)
    return {'llm_answer':response.content}


def action_node(state:InputState):
    messages=[
        ('system',"无论你接收到什么语言的文本，请翻译成英语"),
        ('human',state['llm_answer'])
    ]
    
    llm = ChatOpenAI(
            model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
            api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
            base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
            temperature=0,
        )
    response=llm.invoke(messages)
    return {'answer':response.content}

In [63]:
builder=StateGraph(OverallState,input=InputState,output=OutputState)

builder.add_node('action_node',action_node)
builder.add_node('llm_node',llm_node)

builder.add_edge(START,'llm_node')
builder.add_edge('llm_node','action_node')
builder.add_edge('action_node',END)

graph=builder.compile()

C:\Users\13840\AppData\Local\Temp\ipykernel_8584\3937805902.py:1: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder=StateGraph(OverallState,input=InputState,output=OutputState)
C:\Users\13840\AppData\Local\Temp\ipykernel_8584\3937805902.py:1: LangGraphDeprecatedSinceV05: `output` is deprecated and will be removed. Please use `output_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder=StateGraph(OverallState,input=InputState,output=OutputState)


In [64]:
graph.invoke({'question':'你好，你是谁'})

{'answer': "Hello there! I am your intelligent little assistant, always ready to help you! Whether it's answering questions, offering advice, or just chatting with you, I'm here. Is there anything I can assist you with? 😊"}